# Deploy and Run RMBG 2.0 on Modal

This notebook deploys the RMBG-2.0 background removal model to Modal and runs inference using the deployed class.

In [ ]:
# Deploy the RMBG 2.0 model to Modal
!uv run modal deploy ../llm-hosting/rmbg-2-0.py

In [ ]:
import modal
from PIL import Image
import io

RMBGModel = modal.Cls.from_name("rmbg-2-0", "RMBGModel")
model = RMBGModel()

# Read a test image
image_path = "img.png"
with open(image_path, "rb") as f:
    image_bytes = f.read()

# Run inference remotely (standard generate)
print("Sending request to RMBG 2.0 remote class (standard generate)...")
output_bytes = model.generate.remote(image_bytes)

# Load and display the output image (which has transparency)
output_image = Image.open(io.BytesIO(output_bytes))
output_image.save("rmbg_output.png")
print("Success! Output saved to rmbg_output.png")
output_image

In [ ]:
# Run text-guided object extraction and background removal
print("Sending request to RMBG 2.0 remote class (extract_and_remove_background)...")
# target_object can be a string (e.g. "cup") or None to auto-extract the main object
target_object = None
extracted_bytes = model.extract_and_remove_background.remote(
    image_bytes=image_bytes,
    target_object=target_object
)

# Load and display the output image
extracted_image = Image.open(io.BytesIO(extracted_bytes))
extracted_image.save("extracted_output.png")
print("Success! Isolated object saved to extracted_output.png")
extracted_image